In [1]:
import os
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.pipeline import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.classification import DecisionTreeClassifier

In [2]:
spark = SparkSession.builder.appName('classification').getOrCreate()

In [3]:
display(os.listdir('work'))

['classifer_basic.ipynb',
 'hotel_reservations.csv',
 'hotel_reservations_v1.ipynb',
 'housing_prices.csv',
 'housing_prices_v1.ipynb',
 'housing_prices_v2.ipynb',
 'housing_prices_v3.ipynb',
 'kc_house_data.csv',
 'kc_house_data.ipynb',
 'randomness_latest.csv',
 'recommendation.ipynb']

## Create Dataframe

In [4]:
data = spark.read.csv('work/hotel_reservations.csv', inferSchema=True, header=True)
data.count()

36275

In [5]:
(
   data.select
   (
       [
           F.count(
               F.when(
                   F.col(c).isNull(),1
               )
           ).alias(c) for c in data.columns
       ]
   ).show()
)   

+----------+------------+--------------+--------------------+-----------------+-----------------+--------------------------+------------------+---------+------------+-------------+------------+-------------------+--------------+----------------------------+------------------------------------+------------------+----------------------+--------------+
|Booking_ID|no_of_adults|no_of_children|no_of_weekend_nights|no_of_week_nights|type_of_meal_plan|required_car_parking_space|room_type_reserved|lead_time|arrival_year|arrival_month|arrival_date|market_segment_type|repeated_guest|no_of_previous_cancellations|no_of_previous_bookings_not_canceled|avg_price_per_room|no_of_special_requests|booking_status|
+----------+------------+--------------+--------------------+-----------------+-----------------+--------------------------+------------------+---------+------------+-------------+------------+-------------------+--------------+----------------------------+------------------------------------+--

In [6]:
data.show()

+----------+------------+--------------+--------------------+-----------------+-----------------+--------------------------+------------------+---------+------------+-------------+------------+-------------------+--------------+----------------------------+------------------------------------+------------------+----------------------+--------------+
|Booking_ID|no_of_adults|no_of_children|no_of_weekend_nights|no_of_week_nights|type_of_meal_plan|required_car_parking_space|room_type_reserved|lead_time|arrival_year|arrival_month|arrival_date|market_segment_type|repeated_guest|no_of_previous_cancellations|no_of_previous_bookings_not_canceled|avg_price_per_room|no_of_special_requests|booking_status|
+----------+------------+--------------+--------------------+-----------------+-----------------+--------------------------+------------------+---------+------------+-------------+------------+-------------------+--------------+----------------------------+------------------------------------+--

In [7]:
training_data, testing_data = data.randomSplit([0.8,0.2], seed=42)

## Create Features

In [8]:
pipeline_stages = []

In [9]:
categorical_columns = ['type_of_meal_plan','room_type_reserved','arrival_month','market_segment_type']

for col in categorical_columns:
    string_indexer = StringIndexer(inputCol=col, outputCol=col + '_index')
    print(f'StringIndexer {string_indexer.getInputCol()} -> {string_indexer.getOutputCol()}')

    encoder = OneHotEncoder(inputCol=string_indexer.getOutputCol(), outputCol=col + '_vec', dropLast=False)
    print(f'OneHotEncoder {encoder.getInputCol()} -> {encoder.getOutputCol()}')
    print()
    pipeline_stages += [string_indexer, encoder]

StringIndexer type_of_meal_plan -> type_of_meal_plan_index
OneHotEncoder type_of_meal_plan_index -> type_of_meal_plan_vec

StringIndexer room_type_reserved -> room_type_reserved_index
OneHotEncoder room_type_reserved_index -> room_type_reserved_vec

StringIndexer arrival_month -> arrival_month_index
OneHotEncoder arrival_month_index -> arrival_month_vec

StringIndexer market_segment_type -> market_segment_type_index
OneHotEncoder market_segment_type_index -> market_segment_type_vec



In [10]:
pipeline_stages

[StringIndexer_acc928da3718,
 OneHotEncoder_9609b5ce2460,
 StringIndexer_c05fc220228f,
 OneHotEncoder_6c22f1afcbf6,
 StringIndexer_2cb9ad6cd0af,
 OneHotEncoder_37868b64637a,
 StringIndexer_430097949ab3,
 OneHotEncoder_fc50c9379cb9]

In [11]:
pipeline_stages += [StringIndexer(inputCol='booking_status', outputCol='booking_status_index')]

In [12]:
encoded_categorical_cols = [col + '_vec' for col in categorical_columns]
encoded_categorical_cols

['type_of_meal_plan_vec',
 'room_type_reserved_vec',
 'arrival_month_vec',
 'market_segment_type_vec']

In [13]:
numerical_columns = [
  'no_of_adults',
  'no_of_children',
  'no_of_weekend_nights',
  'no_of_week_nights',
  'required_car_parking_space',
  'lead_time',
  'repeated_guest',
  'no_of_previous_cancellations',
  'no_of_previous_bookings_not_canceled',
  'avg_price_per_room',
  'no_of_special_requests'
]

In [14]:
input_columns = encoded_categorical_cols + numerical_columns
input_columns

['type_of_meal_plan_vec',
 'room_type_reserved_vec',
 'arrival_month_vec',
 'market_segment_type_vec',
 'no_of_adults',
 'no_of_children',
 'no_of_weekend_nights',
 'no_of_week_nights',
 'required_car_parking_space',
 'lead_time',
 'repeated_guest',
 'no_of_previous_cancellations',
 'no_of_previous_bookings_not_canceled',
 'avg_price_per_room',
 'no_of_special_requests']

In [15]:
assembler = VectorAssembler(inputCols=input_columns, outputCol='features')
pipeline_stages.append(assembler)

## Create Algorithm Object

In [16]:
dtc = DecisionTreeClassifier(featuresCol='features', labelCol='booking_status_index')
pipeline_stages.append(dtc)

In [17]:
pipeline_stages

[StringIndexer_acc928da3718,
 OneHotEncoder_9609b5ce2460,
 StringIndexer_c05fc220228f,
 OneHotEncoder_6c22f1afcbf6,
 StringIndexer_2cb9ad6cd0af,
 OneHotEncoder_37868b64637a,
 StringIndexer_430097949ab3,
 OneHotEncoder_fc50c9379cb9,
 StringIndexer_b3c07ee1bb17,
 VectorAssembler_a3c384273cfb,
 DecisionTreeClassifier_75c491e7af64]

## Create  Pipeline and Prediction

In [18]:
pipeline = Pipeline(stages=pipeline_stages)
model = pipeline.fit(training_data)
prediction = model.transform(testing_data)
prediction.select('features','booking_status_index','prediction').show(20)

+--------------------+--------------------+----------+
|            features|booking_status_index|prediction|
+--------------------+--------------------+----------+
|(39,[0,4,21,23,28...|                 1.0|       0.0|
|(39,[0,4,11,23,28...|                 0.0|       0.0|
|(39,[0,4,17,24,28...|                 0.0|       0.0|
|(39,[0,4,16,23,28...|                 1.0|       0.0|
|(39,[0,4,11,24,28...|                 0.0|       0.0|
|(39,[0,4,11,24,28...|                 0.0|       0.0|
|(39,[2,4,12,24,28...|                 0.0|       0.0|
|(39,[1,4,18,23,28...|                 0.0|       0.0|
|(39,[0,4,15,24,28...|                 0.0|       1.0|
|(39,[0,4,16,24,28...|                 0.0|       0.0|
|(39,[2,4,18,24,28...|                 0.0|       0.0|
|(39,[0,5,15,23,28...|                 0.0|       0.0|
|(39,[2,4,18,23,28...|                 1.0|       1.0|
|(39,[0,4,19,23,28...|                 0.0|       0.0|
|(39,[0,4,14,24,28...|                 0.0|       0.0|
|(39,[0,4,

## Create Evaluations

In [19]:
accuracy_evaluator = MulticlassClassificationEvaluator(labelCol='booking_status_index', predictionCol='prediction', metricName='accuracy')
accuracy = accuracy_evaluator.evaluate(prediction) * 100
print(f'Accuracy: {accuracy:.2f}%')

Accuracy: 82.20%


## Create Hyperparameters

In [20]:
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder

In [21]:
paramGrid = (
    ParamGridBuilder()
    .addGrid(dtc.maxDepth, [2,5,10])
    .addGrid(dtc.maxBins, [10,20,40])
    .addGrid(dtc.minInstancesPerNode, [1,2,4])
    .build()
)

evaluator = MulticlassClassificationEvaluator(
    labelCol='booking_status_index',
    predictionCol='prediction',
    metricName='accuracy'
)

tvs_model = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    seed=42
)

best_model = tvs_model.fit(training_data)
prediction = best_model.transform(testing_data)
accuracy = evaluator.evaluate(prediction) * 100
print(f'Best Model Accuracy: {accuracy:.2f}%')

Best Model Accuracy: 86.44%
